# 🎯 Smart Sales & Customer Analytics System

**Projenin Amacı:**
Dağınık halde bulunan müşteri, satış ve bölge tablolarını birleştirerek; eksik verilerden arındırılmış, yönetime karar desteği sunan anlamlı analizler üretmek.

**Pandas Neden Tercih Edilir?**
Tablolar arası ilişkisel (relational) işlemleri (Merge/Join), kayıp veri yönetimini ve kompleks gruplamaları (GroupBy) tek satırda, inanılmaz bir hızla ve esneklikle yapabildiği için.

**Kapsanan Konular:**
* Eksik veriler (isna, fillna, dropna)
* Gruplandırma (groupby, agg)
* Tablo Birleştirme (concat, merge)
* İleri Pandas Operasyonları (apply, koşullu filtreleme)
* Excel Entegrasyonu (to_excel)

In [6]:
import pandas as pd
import numpy as np

print("Veri setleri oluşturuluyor...")

# 1. Customers Tablosu (Yaşlarda NaN var)
customers = pd.DataFrame({
    'CustomerID': [101, 102, 103, 104, 105],
    'CustomerName': ['Ahmet', 'Ayşe', 'Mehmet', 'Fatma', 'Ali'],
    'City': ['İstanbul', 'Ankara', 'İzmir', 'İstanbul', 'Bursa'],
    'Age': [28, np.nan, 35, 42, np.nan]
})

# 2. Sales Tablosu (Tutarlarda NaN var)
sales = pd.DataFrame({
    'SaleID': [1, 2, 3, 4, 5, 6],
    'CustomerID': [101, 102, 101, 104, 105, 103],
    'Product': ['Laptop', 'Mouse', 'Monitor', 'Laptop', 'Keyboard', 'Mouse'],
    'Amount': [15000, 500, np.nan, 16000, 1000, 600]
})

# 3. Regions Tablosu
regions = pd.DataFrame({
    'City': ['İstanbul', 'Ankara', 'İzmir', 'Bursa'],
    'Region': ['Marmara', 'İç Anadolu', 'Ege', 'Marmara']
})

print("✅ Tablolar hazır!")
display(customers.head(), sales.head())

Veri setleri oluşturuluyor...
✅ Tablolar hazır!


,CustomerID,CustomerName,City,Age
0,101,Ahmet,İstanbul,28.0
1,102,Ayşe,Ankara,NaN
2,103,Mehmet,İzmir,35.0
3,104,Fatma,İstanbul,42.0
4,105,Ali,Bursa,NaN


,SaleID,CustomerID,Product,Amount
0,1,101,Laptop,15000.0
1,2,102,Mouse,500.0
2,3,101,Monitor,NaN
3,4,104,Laptop,16000.0
4,5,105,Keyboard,1000.0


In [7]:
print("Eksik veri analizi ve temizliği yapılıyor...\n")

# Eksik verileri tespit et
print("Müşteri tablosundaki eksik veriler:\n", customers.isna().sum())
print("\nSatış tablosundaki eksik veriler:\n", sales.isna().sum())

# Yaş için ortalama ile doldur (fillna)
yas_ortalama = customers['Age'].mean()
customers['Age'] = customers['Age'].fillna(yas_ortalama)

# Satış tutarı eksik olan kayıtları sil (dropna)
sales = sales.dropna(subset=['Amount'])

print("\n✅ Eksik veriler temizlendi!")

Eksik veri analizi ve temizliği yapılıyor...

Müşteri tablosundaki eksik veriler:
 CustomerID      0
CustomerName    0
City            0
Age             2
dtype: int64

Satış tablosundaki eksik veriler:
 SaleID        0
CustomerID    0
Product       0
Amount        1
dtype: int64

✅ Eksik veriler temizlendi!


In [8]:
# Müşteriler ve Satışları CustomerID üzerinden birleştir
merged_df = pd.merge(customers, sales, on='CustomerID', how='inner')

# Çıkan sonucu Bölgeler ile City üzerinden birleştir
final_df = pd.merge(merged_df, regions, on='City', how='inner')

print("✅ Tablolar başarıyla birleştirildi (Merge)!")
display(final_df)

✅ Tablolar başarıyla birleştirildi (Merge)!


,CustomerID,CustomerName,City,Age,SaleID,Product,Amount,Region
0,101,Ahmet,İstanbul,28.0,1,Laptop,15000.0,Marmara
1,102,Ayşe,Ankara,35.0,2,Mouse,500.0,İç Anadolu
2,103,Mehmet,İzmir,35.0,6,Mouse,600.0,Ege
3,104,Fatma,İstanbul,42.0,4,Laptop,16000.0,Marmara
4,105,Ali,Bursa,35.0,5,Keyboard,1000.0,Marmara


In [9]:
# agg() fonksiyonu ile birden fazla analizi tek seferde yapıyoruz
print("📊 GRUPLANDIRMA ANALİZLERİ")

# Şehir bazlı toplam satış
sehir_satis = final_df.groupby('City').agg(Toplam_Satis=('Amount', 'sum'))
print("\n--- Şehir Bazlı Toplam Satış ---")
display(sehir_satis)

# Bölge bazlı ortalama satış
bolge_satis = final_df.groupby('Region').agg(Ortalama_Satis=('Amount', 'mean'))
print("\n--- Bölge Bazlı Ortalama Satış ---")
display(bolge_satis)

# Ürün bazlı satış sayısı
urun_satis = final_df.groupby('Product').agg(Satis_Adedi=('SaleID', 'count'))
print("\n--- Ürün Bazlı Satış Sayısı ---")
display(urun_satis)

📊 GRUPLANDIRMA ANALİZLERİ

--- Şehir Bazlı Toplam Satış ---


,Toplam_Satis
City,
Ankara,500.0
Bursa,1000.0
İstanbul,31000.0
İzmir,600.0



--- Bölge Bazlı Ortalama Satış ---


,Ortalama_Satis
Region,
Ege,600.000000
Marmara,10666.666667
İç Anadolu,500.000000



--- Ürün Bazlı Satış Sayısı ---


,Satis_Adedi
Product,
Keyboard,1
Laptop,2
Mouse,2


In [10]:
# Farklı aylara ait yeni veri simülasyonu
satis_mart = pd.DataFrame({
    'SaleID': [7, 8], 'CustomerID': [102, 103], 
    'Product': ['Tablet', 'Laptop'], 'Amount': [3000, 14000], 'Month': ['Mart', 'Mart']
})

satis_nisan = pd.DataFrame({
    'SaleID': [9, 10], 'CustomerID': [101, 105], 
    'Product': ['Monitor', 'Mouse'], 'Amount': [4500, 700], 'Month': ['Nisan', 'Nisan']
})

# Satır bazlı (alt alta) birleştirme
yeni_satislar = pd.concat([satis_mart, satis_nisan], axis=0, ignore_index=True)

print("✅ Mart ve Nisan satışları Concat ile başarıyla alt alta eklendi!")
display(yeni_satislar)

✅ Mart ve Nisan satışları Concat ile başarıyla alt alta eklendi!


,SaleID,CustomerID,Product,Amount,Month
0,7,102,Tablet,3000,Mart
1,8,103,Laptop,14000,Mart
2,9,101,Monitor,4500,Nisan
3,10,105,Mouse,700,Nisan


In [11]:
# İleri Pandas İşlemleri (final_df üzerinde)

# 1. apply() kullanımı ile Yeni Sütun Oluşturma (KDV Eklenmiş Satış)
# Lambda fonksiyonu ile her bir Amount değerini %20 KDV ile çarpıyoruz
final_df['Amount_with_VAT'] = final_df['Amount'].apply(lambda x: x * 1.20)

# 2. Koşullu Filtreleme (Sadece 5000 TL üzeri yüksek değerli satışlar)
premium_sales = final_df[final_df['Amount'] > 5000]

print("✅ İleri operasyonlar tamamlandı (KDV eklendi ve Premium satışlar filtrelendi)!")
display(premium_sales)

✅ İleri operasyonlar tamamlandı (KDV eklendi ve Premium satışlar filtrelendi)!


,CustomerID,CustomerName,City,Age,SaleID,Product,Amount,Region,Amount_with_VAT
0,101,Ahmet,İstanbul,28.0,1,Laptop,15000.0,Marmara,18000.0
3,104,Fatma,İstanbul,42.0,4,Laptop,16000.0,Marmara,19200.0


In [12]:
# Temizlenmiş ve analiz edilmiş ana tabloyu Excel'e yazdırıyoruz
dosya_adi = "final_sales_report.xlsx"

# Index sütununu (0,1,2,3...) kaydetmemek için index=False yapıyoruz
final_df.to_excel(dosya_adi, index=False, sheet_name="Satis_Analizi")

print(f"📁 Dosya başarıyla oluşturuldu: {dosya_adi}")

📁 Dosya başarıyla oluşturuldu: final_sales_report.xlsx


In [13]:
# Karar Destek Metriklerinin Hesaplanması
toplam_satis = final_df['Amount'].sum()
en_karli_sehir = sehir_satis['Toplam_Satis'].idxmax()
en_guclu_bolge = final_df.groupby('Region')['Amount'].sum().idxmax()

print("="*35)
print("🎯 YÖNETİCİ ÖZETİ")
print("="*35)
print(f"Toplam satış:      {toplam_satis:,.0f} TL")
print(f"En kârlı şehir:    {en_karli_sehir}")
print(f"En güçlü bölge:    {en_guclu_bolge}")
print("="*35)

🎯 YÖNETİCİ ÖZETİ
Toplam satış:      33,100 TL
En kârlı şehir:    İstanbul
En güçlü bölge:    Marmara
